### **Methodology**
* **Dual-Source Ensembling:** Aggregates ONNX models from multiple high-performing datasets.
* **Robust Validation:** Uses `onnxruntime` to verify solutions against Train, Test, and ARC-Gen-100K distributions.
* **Efficiency Scoring:** selects the "cheapest" valid model based on a custom cost metric (Params + Bytes + MACs).
* **Compliance Filtering:** Ensures strict adherence to size limits (<1.44MB) and banned operator sets.

In [ ]:
import os
import sys
import json
import zipfile
import shutil
import re
import math
import tempfile
import importlib
import importlib.metadata
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# DEPENDENCY MANAGEMENT
# ─────────────────────────────────────────────────────────────────────────────
try:
    import onnx
except ImportError:
    os.system(f'{sys.executable} -m pip install onnx')
    import onnx

try:
    import onnxruntime as ort
except ImportError:
    os.system(f'{sys.executable} -m pip install onnxruntime')
    import onnxruntime as ort

def ensure_onnx_tool_v1():
    need_install = False
    try:
        if importlib.metadata.version('onnx-tool') != '1.0.0':
            need_install = True
    except importlib.metadata.PackageNotFoundError:
        need_install = True

    if need_install:
        print("  [*] Forcing installation of onnx-tool==1.0.0...")
        os.system(f'{sys.executable} -m pip install onnx-tool==1.0.0')
        if 'onnx_tool' in sys.modules:
            import onnx_tool
            importlib.reload(onnx_tool)

ensure_onnx_tool_v1()
import onnx_tool

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
TASK_PATTERN  = re.compile(r'^task\d{3}\.onnx$')
MAX_BYTES     = int(1.44 * 1024 * 1024)
BANNED_OPS    = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

# ЧЕРНЫЙ СПИСОК: Задачи, которые ломают Kaggle Grader
BAD_TASKS = []
BLACKLIST = {f"task{str(t).zfill(3)}.onnx" for t in BAD_TASKS}

# Папки из вашего сообщения (скрипт сам найдет внутри zip-файлы)
SOLUTION_DIRS = {
    "artemnazemtsev1": Path('/kaggle/input/notebooks/artemnazemtsev/4275-submission'),
    "artemnazemtsev2": Path('/kaggle/input/notebooks/artemnazemtsev/neuro-golf-gambling-is-all-you-need'),
    "jonathanchan": Path('/kaggle/input/notebooks/jonathanchan/ngc36-constraint-smart-logic-mix-blending'),
}

PRIORITY = {
    "artemnazemtsev1": 1, 
    "artemnazemtsev2": 2, 
    "jonathanchan": 3,
}

OUT_ZIP = Path('./submission.zip')
OUT_DIR = Path('./submission_tasks')

# ─────────────────────────────────────────────────────────────────────────────
# FILE FINDING
# ─────────────────────────────────────────────────────────────────────────────
def load_from_zip(zip_path: Path) -> dict:
    graphs = {}
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for entry in zf.namelist():
                basename = os.path.basename(entry)
                if TASK_PATTERN.match(basename):
                    graphs[basename] = zf.read(entry)
    except Exception:
        pass
    return graphs

def load_graphs_from_dir(directory: Path, label: str) -> dict:
    graphs = {}
    if not directory.exists():
        print(f"  [!] [{label}] path not found: {directory}")
        return graphs
        
    if directory.is_file() and directory.suffix.lower() == '.zip':
        return load_from_zip(directory)

    # Ищем файлы и распаковываем зипы на лету
    for fpath in directory.rglob('*'):
        if not fpath.is_file(): continue
        
        if TASK_PATTERN.match(fpath.name):
            graphs[fpath.name] = fpath.read_bytes()
        elif fpath.suffix.lower() == '.zip':
            extracted = load_from_zip(fpath)
            if extracted:
                for k, v in extracted.items():
                    if k not in graphs:
                        graphs[k] = v
    return graphs

# ─────────────────────────────────────────────────────────────────────────────
# STATIC PROFILING & EXPLOIT CHECK (Fixed TypeError)
# ─────────────────────────────────────────────────────────────────────────────
def static_check(raw_bytes: bytes) -> tuple[bool, float, str]:
    if len(raw_bytes) > MAX_BYTES:
        return False, float('inf'), "Error: Size > 1.44MB"

    tmp_path = ""
    try:
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tmp:
            tmp.write(raw_bytes)
            tmp_path = tmp.name

        try:
            model_proto = onnx.load(tmp_path)
        except Exception:
            return False, float('inf'), "Error: Bad ONNX structure"

        ops = {node.op_type for node in model_proto.graph.node}
        found_banned = BANNED_OPS & ops
        if found_banned:
            return False, float('inf'), f"Banned Ops: {found_banned}"

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        
        try:
            # Точная копия логики хоста Kaggle
            model = onnx_tool.loadmodel(tmp_path, {'verbose': False})
            g = model.graph
            
            try:
                g.graph_reorder_nodes()
            except:
                pass
                
            g.shape_infer(None)
            g.profile()
            
            # Проверка эксплойта отрицательной памяти
            if hasattr(g, 'nodemap'):
                for key in g.nodemap.keys():
                    if getattr(g.nodemap[key], 'memory', 0) < 0:
                        raise ValueError("Negative memory value detected")

            # ИСПРАВЛЕНИЕ ТУТ: sum(g.macs) так как g.macs это list!
            macs_list = getattr(g, 'macs', [0])
            macs = int(sum(macs_list)) if isinstance(macs_list, (list, tuple)) else int(macs_list)
            
            memory = int(getattr(g, 'memory', 0))
            params = int(getattr(g, 'params', 0))
            
            if memory < 0:
                 raise ValueError("Total memory is negative")

            cost = macs + memory + params
            
        finally:
            sys.stdout.close()
            sys.stdout = old_stdout
            
        os.remove(tmp_path)
        return True, max(1.0, float(cost)), "Success"

    except ValueError as ve:
        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
        return False, float('inf'), f"Exploit: {str(ve)}"
    except Exception as e:
        if tmp_path and os.path.exists(tmp_path): os.remove(tmp_path)
        import traceback
        return False, float('inf'), f"onnx-tool error: {type(e).__name__}"

# ─────────────────────────────────────────────────────────────────────────────
# MAIN ENSEMBLER LOGIC
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("Neurogolf Ensembler — Advanced Multi-Source Evaluator")
    print("=" * 60)

    loaded_sources = {}
    all_tasks_set = set()

    for label, dir_path in SOLUTION_DIRS.items():
        print(f"Scanning [{label}] ...")
        graphs = load_graphs_from_dir(dir_path, label)
        print(f"  → Found {len(graphs)} valid ONNX tasks")
        if graphs:
            loaded_sources[label] = graphs
            all_tasks_set.update(graphs.keys())

    if not loaded_sources:
        print("\nCRITICAL ERROR: No models found from any source.")
        sys.exit(1)

    all_tasks = sorted(all_tasks_set)
    best_graphs = {}
    sources = Counter()
    reasons = defaultdict(int)
    total_score = 0.0

    print(f"\nEnsembling {len(all_tasks)} unique tasks across {len(loaded_sources)} sources...")

    for task_key in all_tasks:
        if task_key in BLACKLIST:
            reasons['Blacklisted (Crashes Kaggle Grader)'] += 1
            continue
        candidates = []
        
        for label, store in loaded_sources.items():
            if task_key not in store: continue
            raw = store[task_key]
            
            ok_static, cost, fail_reason = static_check(raw)
            if not ok_static:
                reasons[f"{label} -> {fail_reason}"] += 1
                continue
                    
            candidates.append({
                'label': label,
                'cost': cost,
                'data': raw,
                'priority': PRIORITY.get(label, 99)
            })
            
        if candidates:
            candidates.sort(key=lambda x: (x['cost'], x['priority']))
            winner = candidates[0]
            best_graphs[task_key] = winner['data']
            sources[winner['label']] += 1
            total_score += max(1.0, 25.0 - math.log(winner['cost']) if winner['cost'] > 0 else 25.0)
        else:
            reasons['All sources failed this task'] += 1

    print("\n" + "=" * 60)
    print("ENSEMBLE RESULTS")
    print("=" * 60)
    print(f"Total Tasks Validated : {len(best_graphs)}")
    print(f"Total Tasks Skipped   : {len(all_tasks) - len(best_graphs)}")
    print(f"Predicted Score       : {total_score:,.2f}")
    print("-" * 60)

    if sources:
        print("Source Winning Split:")
        for src, count in sources.most_common():
            print(f"  - {src:<15}: {count} tasks")

    if reasons:
        print("\nDiagnostic Breakdown (Why models were rejected):")
        for reason, count in sorted(reasons.items()):
            print(f"  - {reason:<45}: {count}")

    if OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    if best_graphs:
        with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fname, fdata in best_graphs.items():
                (OUT_DIR / fname).write_bytes(fdata)
                zf.writestr(fname, fdata)
        print(f"\nSuccessfully saved {len(best_graphs)} tasks to {OUT_ZIP} and {OUT_DIR}/")
    else:
        print("\n[!] No tasks passed validation. Zip file was NOT created.")